# 01 — Hello World Agent

**Mode:** Offline  
**Level:** Fundamentals  
**Estimated time:** 25 minutes

[Watch the original lesson video](https://www.youtube.com/watch?v=X25zLFSmXt8)

This notebook is meant to be read and run from top to bottom. Each code cell
changes one small part of the system, and the inspection cells show the real
Praval objects produced by that change.


## What you will build

A two-agent greeting workflow. A decorated greeter reacts to input, emits a new Spore, and a listener captures the output. You will also construct an Agent directly and compare both styles.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "examples" / "notebooks",
]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    find_example_asset, get_events, require_env, setup_notebook,
    show_callout, show_flow, show_json, show_roles, show_spore,
    show_timeline, stage,
)

setup_notebook(1, 'Hello World Agent')


## Prerequisites and setup

**Course prerequisites:** Complete `course-00-architecture`.

**Execution requirements:** Offline. Complete lesson 00 first; no API key is required.

Run the setup cell above first. It configures presentation helpers and a
credential-free lifecycle provider. It does not hide any agent workflow.


## Learning goals

- Create agents with the `@agent` decorator and with `Agent(...)`.
- Use identity and `responds_to` to define an agent's boundary.
- Trace inputs, outputs, and cascading delivery.
- Close agents and shut down their Reef.


## Mental model

The decorator turns a handler function into a managed Agent while keeping the function readable. Direct construction gives you the Agent object first. In both forms, **identity** names the participant, `responds_to` filters message types, and lifecycle methods release runtime resources.


In [ ]:
show_flow(
('Notebook', 'greeting input', 'human'),
('Greeter', 'build response', 'agent'),
('Reef', 'route response', 'reef'),
('Listener', 'collect output', 'agent'),
)


## Try it

We will now assemble the workflow in small steps. Run each cell, then pause at the inspection that follows it.


### Reset the shared Reef

Decorated agents use Praval's shared Reef. Resetting it makes re-running this notebook behave like a clean application start.


In [ ]:
from praval import Agent, agent, broadcast, get_reef, start_agents
from praval.core.reef import reset_reef

reset_reef()
observed = []


### What just happened?

The global Reef has no stale channels or subscriptions from an earlier run.

### Why this matters

Notebook cells are often re-run. Explicit reset and cleanup prevent duplicate delivery from older agent registrations.


### Define the greeter

The handler accepts only Spores whose knowledge type is `greeting`. `auto_broadcast=False` means the function decides what to publish.


In [ ]:
@agent("course-greeter", responds_to=["greeting"], auto_broadcast=False)
def greeter(spore):
    stage("greeter", "received", spore.knowledge["type"], spore)
    name = spore.knowledge.get("name", "World")
    broadcast(
        {"type": "greeting_response", "message": f"Hello, {name}!"}
    )
    stage("greeter", "broadcast", "greeting_response", spore)


### What just happened?

The decorator attached a managed Agent at `greeter._praval_agent`; the function body is still the visible message handler.

### Why this matters

A narrow `responds_to` list documents the input contract and prevents the agent from reacting to unrelated traffic.


In [ ]:
show_json(
    {
        "decorated_name": greeter._praval_agent.name,
        "responds_to": ["greeting"],
        "callable": greeter.__name__,
    },
    "Greeter identity",
)


### Define the listener and a direct Agent

The listener is another decorated handler. The direct Agent is included to show the alternative construction style without adding it to this route.


In [ ]:
@agent("course-listener", responds_to=["greeting_response"], auto_broadcast=False)
def listener(spore):
    stage("listener", "received", spore.knowledge["message"], spore)
    observed.append(spore)


direct_agent = Agent(
    "course-direct-agent",
    provider="notebook-lifecycle",
    model="notebook-lifecycle-model",
)


### What just happened?

Both construction styles produced Agent objects with stable names. Only the decorated listener is subscribed to greeting responses.

### Why this matters

Use decorators for message handlers and direct construction when you want to configure or invoke an Agent object explicitly.


In [ ]:
show_roles([
    (greeter._praval_agent.name, "Transform greeting input", "agent"),
    (listener._praval_agent.name, "Collect greeting output", "agent"),
    (direct_agent.name, "Direct construction example", "agent"),
])


### Start the workflow

`start_agents()` registers both decorated agents, then injects one initial knowledge dictionary on a named channel.


In [ ]:
start_agents(
    greeter,
    listener,
    initial_data={"type": "greeting", "name": "Praval learner"},
    channel="course-hello-world",
)
reef = get_reef()
assert reef.wait_for_completion(timeout=10)

messages = [item.knowledge["message"] for item in observed]
assert messages == ["Hello, Praval learner!"]


### What just happened?

The initial greeting reached only the greeter. Its response created a second Spore, which reached only the listener. Completion included both deliveries.

### Why this matters

Cascading messages let a workflow grow without one function directly calling every later step.


In [ ]:
show_spore(observed[0], "Greeting response received by the listener")
show_timeline()
show_json(reef.get_network_stats(), "Shared Reef state")


## Your turn

Change the initial name and add a `language` field. Update the greeter to include that field in the response. Verify the listener observes it.


In [ ]:
# initial = {"type": "greeting", "name": "Ada", "language": "English"}
# In greeter(), read spore.knowledge["language"] and include it in the response.
# Re-run from the reset cell so each agent is registered exactly once.


## Common mistake

**Mistake:** Re-running only the decorator cells and leaving old agents registered.

**Correction:** Run cleanup or reset the Reef before redefining agents with the same names.


<details>
<summary>Under the hood</summary>

The decorator stores its Agent object on `_praval_agent`. `start_agents()` registers their handlers with the shared Reef and broadcasts the initial payload. `responds_to` filtering happens before the handler body runs.

</details>


## Recap

- Agent identity is a stable name used throughout the workflow.
- Decorated and direct construction serve different ergonomic needs.
- Handlers receive Spores and can publish the next stage.
- Lifecycle cleanup matters in notebooks and long-running applications.


## Cleanup

Release every agent, transport, temporary file, or external resource created by this lesson. Cleanup is part of the example, not an afterthought.


In [ ]:
direct_agent.close()
greeter._praval_agent.close()
listener._praval_agent.close()
assert reef.shutdown(timeout=10)
stage("reef", "shutdown", "agents and channels closed")
show_timeline()


### Next lesson

Continue to `02_research_pipeline.ipynb` to connect two model-backed agents through an intermediate findings Spore.
